# Stage 10 — Basic Chatbot Experience in Colab

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** A pleasant, repeatable way to chat with ResearchMate directly in Colab, plus a categorized automated test run — before we build the Streamlit UI.

**Before running:**
1. Upload `faiss_index.zip` (from Stage 6).
2. Make sure your `GROQ_API_KEY` Colab Secret is still set up.

## Cell 1 — Install packages

In [1]:
!pip install -q langchain-groq faiss-cpu langchain-community langchain-huggingface sentence-transformers langchain-core pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Cell 2 — Rebuild the full pipeline (from Stage 9)

Same setup as Stage 9: API key, vector store, retriever, LLM, prompt, `format_docs`, `answer_chain`, `RAGResponse`, and `ask_researchmate()`.

In [2]:
import os
import zipfile
from google.colab import userdata
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import List

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

with zipfile.ZipFile("faiss_index.zip", "r") as zip_ref:
    zip_ref.extractall("faiss_index")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

SYSTEM_PROMPT = """You are ResearchMate, a research paper assistant.
Answer the user's question using ONLY the research paper context provided below.

Rules:
- Do not invent or assume any information that is not present in the context.
- If the context does not contain enough information to answer the question, say so clearly instead of guessing.
- When you use information from a paper, mention its title.
- Keep your answer clear and concise.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(f"Title: {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

answer_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

class RAGResponse(BaseModel):
    answer: str
    relevant_papers: List[str]
    topics: List[str]

def ask_researchmate(question):
    docs = retriever.invoke(question)
    answer_text = answer_chain.invoke(question)
    relevant_papers = [doc.metadata["title"] for doc in docs]
    all_topics = set()
    for doc in docs:
        for topic in doc.metadata["topics"].split(", "):
            if topic:
                all_topics.add(topic)
    return RAGResponse(answer=answer_text, relevant_papers=relevant_papers, topics=sorted(all_topics))

print("Pipeline ready.")

/tmp/ipykernel_2233/3962361819.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pipeline ready.


## Cell 3 — A nicer print helper

Formats one `RAGResponse` into a clean, readable block instead of raw Python object output.

In [3]:
def pretty_print(question, response):
    print(f"You asked: {question}\n")
    print(f"ResearchMate: {response.answer}\n")
    print("Relevant papers:")
    for title in response.relevant_papers:
        print(f"  - {title}")
    print(f"Topics: {', '.join(response.topics)}")
    print("-" * 70)

print("pretty_print() ready.")

pretty_print() ready.


## Cell 4 — Interactive chat loop

Run this cell, then type questions at the prompt. Type `exit` or `quit` to stop. Every question and its response are also saved into `chat_history` as you go, so you can review the whole session afterward.

In [4]:
chat_history = []

def chat_loop():
    print("ResearchMate is ready. Ask a question about research papers, or type 'exit' to stop.\n")
    while True:
        question = input("You: ").strip()
        if question.lower() in ("exit", "quit"):
            print("Session ended.")
            break
        if not question:
            continue
        response = ask_researchmate(question)
        chat_history.append((question, response))
        pretty_print(question, response)

chat_loop()

ResearchMate is ready. Ask a question about research papers, or type 'exit' to stop.

You: exit
Session ended.


## Cell 5 — Automated categorized test run

Since a typed interactive session doesn't leave output you can paste back, this cell runs a fixed set of questions automatically across 3 categories: common research questions, specific/technical questions, and questions with little-to-no matching content in our dataset. This gives us a reproducible record of how ResearchMate handles each kind of input.

In [5]:
test_suite = {
    "Common research questions": [
        "What research has been done on deep learning?",
        "What work exists on Bayesian statistics?",
    ],
    "Specific/technical questions": [
        "What are recent approaches to graph neural network evaluation?",
        "How is the Frechet distribution estimated using Bayesian methods?",
    ],
    "Little-to-no matching information": [
        "What is the best recipe for making butter chicken?",
        "What research has been done on transformer efficiency?",
    ],
}

for category, questions in test_suite.items():
    print(f"### {category} ###\n")
    for q in questions:
        response = ask_researchmate(q)
        chat_history.append((q, response))
        pretty_print(q, response)
    print()

### Common research questions ###

You asked: What research has been done on deep learning?

ResearchMate: Research on deep learning has covered a range of topics and applications:

1. **Historical and Critical Review** – *Deep Learning: A Critical Appraisal* surveys the field’s evolution since the 2012 Imagenet breakthrough, highlighting progress in speech, image, and game domains while raising ten concerns that suggest deep learning alone may not achieve artificial general intelligence.

2. **Domain‑Specific Applications** – *A Survey of Deep Learning Techniques for Mobile Robot Applications* reviews how deep neural networks are being integrated into robotic systems, summarizing the benefits and challenges of applying deep learning to mobile robotics.

3. **Practical Guidance for New Users** – *Best Practices for Applying Deep Learning to Novel Applications* offers a phased approach for subject‑matter experts who are new to deep learning, providing recommendations and insights to hel

## What to check after running this notebook

- **Cell 4:** try a few questions of your own interactively — does the experience feel smooth and easy to demo?
- **Cell 5 — Common questions:** should get solid, well-grounded answers like earlier stages.
- **Cell 5 — Specific/technical questions:** check whether the more narrowly-worded questions still retrieve genuinely matching papers.
- **Cell 5 — Little-to-no-match questions:** confirm the butter chicken question gets an honest "not relevant" answer (not a hallucinated one), and see how it handles transformer efficiency now.

Paste back the full output of Cell 5 (the automated test suite) — then we'll move to Stage 11 (Streamlit UI).